In [ ]:
import numpy as np
import pandas as pd
import yaml
import re
import glob
from tqdm import tqdm

In [ ]:
import os
os.chdir('../../')

In [ ]:
def pretty_convert(filename):
    # Extract the part after 'recognized_by_' and before '.parquet'
    m = re.search(r'recognized_by_(.*?)\.parquet$', filename, re.IGNORECASE)
    if not m:
        return None
    label = m.group(1)
    
    # Special handling: remove trailing '_match' if it exists
    if label.lower().endswith('_match'):
        label = label[:-len('_match')]
        
    # Replace underscores with spaces
    label = label.replace('_', ' ')
    
    # Title-case the label unless it is all uppercase (e.g., "LLM")
    if not label.isupper():
        label = label.title()
        
    return label

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
result_parquet = ['OA3_recognized_by_openalex.parquet',
                  'POST1_recognized_by_country_name.parquet',
                  'POST1_recognized_by_top_uni_company.parquet',
                  'POST1_recognized_by_city_state_match.parquet',
                  'POST2_recognized_by_LLM.parquet']

In [ ]:
dfs_result = []

for parquet_file in result_parquet:
    print('Reading:', parquet_file)
    
    df_temp = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/' +  parquet_file)
    source = pretty_convert(parquet_file)
    df_temp['source'] = source

    dfs_result.append(df_temp)

In [ ]:
recognized_all = pd.concat(dfs_result).drop(columns=['affiliationame'])
recognized_all

In [ ]:
result_source = recognized_all.value_counts('source')
result_source

In [ ]:
for idx in range(len(result_source)):
    source_name = result_source.index[idx]
    source_count = result_source.iloc[idx]
    print(f'{source_name:40s} {source_count:10d} {source_count / 9887831 * 100:15.4f} %')
print('-'*70)
print(f"{'TOTAL':40s} {recognized_all.shape[0]:10d} {recognized_all.shape[0] / 9887831 * 100:15.4f} %")

### Keep only the last author country info in each patent-work pair

In [ ]:
ROS_last = recognized_all[recognized_all['author_position'] == -1][['paperid', 'patent_id', 'country']].drop_duplicates()
ROS_last

### Process OA paper_year

In [ ]:
# Create a list of all .csv.gz files in the directory
file_pattern = os.path.join(dataset_config['path_openalex'], 'works_year_*.csv.gz')
csv_files = glob.glob(file_pattern)

# Read each file into a DataFrame and store them in a list
dataframes = []
for file in tqdm(csv_files):
    try:
        df = pd.read_csv(file, compression='gzip')
        if len(df) >= 1:
            dataframes.append(df.rename(columns={'work_id': 'paperid'}))
    except Exception as e:
        print(f"Error reading {file}: {e}")

In [ ]:
oa_year = pd.concat(dataframes, ignore_index=True).rename(columns={'publication_year': 'year'})
oa_year

In [ ]:
del dataframes

In [ ]:
ROS_results = ROS_last.merge(oa_year, on='paperid')
ROS_results

In [ ]:
ROS_results.to_parquet(dataset_config['path_processed'] + 'CN_CN/POST3_ROS_results.parquet', index=False)